In [ ]:
# ===== Cell 1 =====
import pandas as pd  # 데이터 처리
import numpy as np  # 수치 계산
import re  # 정규표현식
import random  # 랜덤 샘플링 사용

In [ ]:
PATH_SUFUL = r"C:/Users/User/OneDrive/문서/3yejoo/ERP/재고/251231_재고조사/251231_본사재고수불부모음_ver6_수정.xlsx"  # 수불부
PATH_SUBMIT = r"C:/Users/User/OneDrive/문서/3yejoo/ERP/재고/251231_재고조사/251231_본사제출용.xlsx"  # 제출용

# 엑셀 파일 읽기
df_submit = pd.read_excel(PATH_SUBMIT)  # 본사제출용 로드
df_suful = pd.read_excel(PATH_SUFUL)  # 수불부모음 로드

In [ ]:
# ===== Cell 2 =====
WAREHOUSE_KEYWORD = "본사"  # 본사 키워드
def normalize_item_code(code: str) -> str:  # 품목코드 표준화
    s = str(code).strip()  # 공백 제거
    s = re.sub(r"\.0$", "", s)  # 끝에 .0 제거
    if re.fullmatch(r"\d+", s):  # 숫자만이면
        s = s.lstrip("0")  # 앞 0 제거
        return s if s != "" else "0"  # 전부 0이면 0
    return s  # 문자 포함 코드는 그대로


df_suful["일자"] = df_suful["일자"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()  # 일자 공백 정리
df_suful["일자"] = pd.to_datetime(df_suful["일자"], errors="coerce", format="mixed")  # 일자 변환
df_suful = df_suful.dropna(subset=["일자"]).copy()  # 변환 실패 제거
df_suful["월"] = df_suful["일자"].dt.to_period("M").astype(str)  # 월 컬럼 생성

df_suful["품목코드"] = df_suful["품목코드"].astype(str).apply(normalize_item_code)  # 수불부 품목코드 정리
df_submit["품목코드"] = df_submit["품목코드"].astype(str).apply(normalize_item_code)  # 제출용 품목코드 정리

df_suful["재고수량"] = pd.to_numeric(df_suful["재고수량"], errors="coerce").fillna(0).astype(float)  # 재고수량
df_suful["출고단가"] = pd.to_numeric(df_suful["출고단가"], errors="coerce").fillna(0).astype(float)  # 출고단가
df_suful["출고수량"] = pd.to_numeric(df_suful["출고수량"], errors="coerce").fillna(0).astype(float)  # 출고수량

df_suful = df_suful[df_suful["창고명"].astype(str).str.contains(WAREHOUSE_KEYWORD, na=False)].copy()  # 본사만
df_suful = df_suful[df_suful["거래처명"].astype(str).str.strip() != "[조정]"].copy()  # [조정] 제외


item_codes = df_submit["품목코드"].dropna().unique().tolist()  # 제출용 품목코드 리스트
n = min(100, len(item_codes))  # 최대 100개
sample_codes = random.sample(item_codes, n)  # 랜덤 추출

df_suful_100 = df_suful[df_suful["품목코드"].isin(sample_codes)].copy()  # 수불부 필터링

# ===== Cell 3 =====
OUT_SUFUL_PATH = r"./df_suful_100.xlsx"  # 수불부 샘플 저장 경로
OUT_SUBMIT_PATH = r"./df_submit_100.xlsx"  # 제출용 샘플 저장 경로

# 제출용에서 선택된 품목코드만 필터링
df_submit_100 = df_submit[df_submit["품목코드"].isin(sample_codes)].copy()  # 제출용 100개만

# 수불부 샘플 저장
df_suful_100.to_excel(OUT_SUFUL_PATH, index=False)  # 엑셀로 저장

# 제출용 샘플 저장
df_submit_100.to_excel(OUT_SUBMIT_PATH, index=False)  # 엑셀로 저장

print("저장 완료")  # 완료 로그
print("수불부 행 수:", len(df_suful_100))  # 수불부 행 수
print("제출용 행 수:", len(df_submit_100))  # 제출용 행 수

In [24]:
# ===== Cell 5 =====
# 샘플플엑셀 파일 읽기
df_submit_100 = pd.read_excel("df_submit_100.xlsx")  # 본사제출용 로드
df_suful_100 = pd.read_excel("df_suful_100.xlsx")  # 수불부모음 로드

In [25]:
# ===== Cell 6 =====
# 품목코드 + 일자 기준 정렬
df_suful_100 = df_suful_100.sort_values(
    ["품목코드", "일자"]
).copy()  # 순서 보장

# 재고수량이 마이너스인 행만 필터링
df_minus = df_suful_100[df_suful_100["재고수량"] < 0].copy()  # 중간 마이너스

# 품목코드별로 최초 마이너스 발생 지점만 추출
df_first_minus = (
    df_minus
    .groupby("품목코드", as_index=False)
    .first()
)  # 품목별 첫 마이너스

# 가장 먼저 마이너스가 발생한 품목 1개 선택
target_item = df_first_minus.sort_values("일자").iloc[0]["품목코드"]  # 품목코드 선택

print("선택된 품목코드:", target_item)  # 선택 결과 출력

# 선택된 품목 전체 이력 조회
df_target = df_suful_100[df_suful_100["품목코드"] == target_item].copy()  # 해당 품목만

# 선택된 품목만 가져와서 정렬
df_target = (
    df_suful_100[df_suful_100["품목코드"] == target_item]
    .sort_values("일자")
    .reset_index(drop=True)
)  # 품목 단위 정렬

# 마이너스 여부 판단
df_target["is_minus"] = df_target["재고수량"] < 0  # 마이너스 여부

# 상태가 바뀔 때마다 그룹 번호 증가
df_target["group"] = df_target["is_minus"].ne(df_target["is_minus"].shift()).cumsum()  # 구간 그룹

# 마이너스 구간만 추출
df_minus = df_target[df_target["is_minus"]].copy()  # 마이너스만

# 구간별 요약 생성
result = (
    df_minus
    .groupby("group")
    .agg(
        시작일=("일자", "min"),          # 구간 시작일
        종료일=("일자", "max"),          # 구간 종료일
        최대마이너스수=("재고수량", "min")  # 구간 최저 재고
    )
    .reset_index(drop=True)
)  # 최종 결과

print(result)  # 결과 출력

선택된 품목코드: 3089
         시작일        종료일  최대마이너스수
0 2025-01-02 2025-02-05      -11
1 2025-02-27 2025-03-18      -10
2 2025-04-01 2025-04-01       -1
3 2025-10-17 2025-10-17       -2


In [32]:
# ===== Cell 10 =====
# 선택된 품목만 가져와서 정렬
df_target = (  # 대상 품목만
    df_suful_100[df_suful_100["품목코드"].astype(str) == str(target_item)]  # 품목 필터
    .sort_values("일자")  # 날짜 정렬
    .reset_index(drop=True)  # 인덱스 초기화
)  # 대상 이력

# 마이너스 여부 판단
df_target["is_minus"] = df_target["재고수량"] < 0  # 마이너스 여부

# 상태 변화 기준으로 구간 번호 생성
df_target["group"] = df_target["is_minus"].ne(df_target["is_minus"].shift()).cumsum()  # 구간 그룹

# 마이너스 구간만 추출
df_minus = df_target[df_target["is_minus"]].copy()  # 마이너스만

# 구간별 최저점(최대마이너스) 행 인덱스 추출
idx_min = df_minus.groupby("group")["재고수량"].idxmin()  # 구간별 최저점 인덱스

# 구간별 요약 생성
result = (  # 구간 요약
    df_minus.groupby("group")  # 구간별
    .agg(  # 집계
        시작일=("일자", "min"),  # 구간 시작
        종료일=("일자", "max"),  # 구간 종료
        최대마이너스수=("재고수량", "min"),  # 구간 최저 재고
    )  # 집계 끝
    .reset_index()  # 인덱스 리셋
)  # 요약 테이블

# 최대마이너스 발생일 추가
result["최대마이너스발생일"] = result["group"].apply(lambda g: df_target.loc[idx_min[g], "일자"])  # 최저점 날짜

# 최대마이너스 발생일의 출고수량 추가
result["출고수량"] = result["group"].apply(lambda g: df_target.loc[idx_min[g], "출고수량"])  # 최저점 출고수량

# 첫 번째 마이너스 구간을 기준으로 보정 날짜/수량 결정
fix_row = result.sort_values("시작일").iloc[0]  # 가장 이른 구간 1개
TARGET_DATE = pd.Timestamp(fix_row["시작일"])  # 보정 날짜(구간 시작일)
NEED_QTY = int(abs(fix_row["최대마이너스수"]))  # 보정 수량(최대마이너스 절대값)

# 제출본 컬럼명(업로드된 df_submit_100 기준)
stock_col_submit = "본사실재고"  # 제출본 실재고 컬럼
use_col_submit = "사용여부"  # 제출본 과세/면세 컬럼
name_col_submit = "품목명"  # 제출본 품목명 컬럼

# 타입 정리
df_submit_100["품목코드"] = df_submit_100["품목코드"].astype(str)  # 제출본 코드 문자열
df_suful_100["품목코드"] = df_suful_100["품목코드"].astype(str)  # 수불부 코드 문자열
df_submit_100[stock_col_submit] = pd.to_numeric(df_submit_100[stock_col_submit], errors="coerce").fillna(0).astype(float)  # 실재고 숫자화

# 선택품목(제출본) 정보
target_row_submit = df_submit_100[df_submit_100["품목코드"] == str(target_item)].iloc[0]  # 선택품목 1행
target_stock_submit = float(target_row_submit[stock_col_submit])  # 선택품목 제출 실재고
target_use = str(target_row_submit[use_col_submit]).strip()  # 선택품목 과세/면세
target_name = str(target_row_submit[name_col_submit]).strip()  # 선택품목 품목명

# 전체 정렬(조회용)
df_suful_100 = df_suful_100.sort_values(["품목코드", "일자"]).copy()  # 전체 정렬

def get_stock_asof(item_code: str, date: pd.Timestamp) -> float:  # 특정일 기준 재고(당일 없으면 직전)
    d = df_suful_100[df_suful_100["품목코드"] == item_code].copy()  # 해당 품목만
    d = d[d["일자"] <= date].copy()  # 해당일 이전/당일
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d["재고수량"], errors="coerce").fillna(np.nan).iloc[-1])  # 마지막 재고

def get_out_price_asof(item_code: str, date: pd.Timestamp) -> float:  # 특정일 출고단가(당일 없으면 직전 단가)
    d = df_suful_100[df_suful_100["품목코드"] == item_code].copy()  # 해당 품목만
    d = d[d["일자"] <= date].copy()  # 해당일 이전/당일
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d["출고수량"] = pd.to_numeric(d["출고수량"], errors="coerce").fillna(0)  # 출고수량 숫자화
    d["출고단가"] = pd.to_numeric(d["출고단가"], errors="coerce")  # 출고단가 숫자화
    d = d[(d["출고수량"] > 0) & (d["출고단가"] > 0)].copy()  # 출고+단가 있는 행만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d_day = d[d["일자"] == date].copy()  # 당일만
    if len(d_day) > 0:  # 당일 단가 있으면
        return float(d_day["출고단가"].mean())  # 당일 평균 단가
    return float(d["출고단가"].iloc[-1])  # 직전 단가

# 선택품목 출고단가(비교용) 계산
target_unit_price = get_out_price_asof(str(target_item), TARGET_DATE)  # 선택품목 단가
if pd.isna(target_unit_price):  # 단가 없으면
    raise ValueError("선택품목 출고단가를 TARGET_DATE 기준으로 못 찾아서 단가 비교 불가")  # 중단

# 후보 시작(제출본 기준)
cand = df_submit_100.copy()  # 후보 복사
cand = cand[cand["품목코드"] != str(target_item)].copy()  # 자기 자신 제외
cand = cand[cand[stock_col_submit] > target_stock_submit].copy()  # (1) 제출 실재고가 더 많은 품목
cand = cand[cand[use_col_submit].astype(str).str.strip() == target_use].copy()  # (2) 과세/면세 동일

# 1/2 기준 재고 계산
cand["재고_T"] = cand["품목코드"].apply(lambda x: get_stock_asof(str(x), TARGET_DATE))  # T일 재고
cand["보정출고후재고_T"] = cand["재고_T"] - NEED_QTY  # T일에 NEED_QTY 추가 출고 가정
cand = cand[cand["보정출고후재고_T"] > 0].copy()  # (3) 보정 후 재고가 0보다 커야 함

# 1/2 기준 단가 계산
cand["출고단가_T"] = cand["품목코드"].apply(lambda x: get_out_price_asof(str(x), TARGET_DATE))  # T일 단가(없으면 직전)
cand = cand[~pd.isna(cand["출고단가_T"])].copy()  # 단가 없는 품목 제외

# 단가 차이율 계산(10% 이내)
cand["단가차이율"] = (cand["출고단가_T"] - float(target_unit_price)).abs() / float(target_unit_price)  # 차이율
cand = cand[cand["단가차이율"] <= 0.2].copy()  # (4) 10% 이내

# 최종 대체품목 리스트 생성
substitute_list = cand[  # 결과 컬럼만
    ["품목코드", name_col_submit, use_col_submit, stock_col_submit, "재고_T", "보정출고후재고_T", "출고단가_T", "단가차이율"]  # 출력 컬럼
].copy()  # 복사

# 보기 좋게 정렬
substitute_list = substitute_list.sort_values(["단가차이율", stock_col_submit], ascending=[True, False]).reset_index(drop=True)  # 정렬

# 결과 출력
print("선택품목코드:", target_item)  # 선택 품목코드
print("선택품목명:", target_name)  # 선택 품목명
print("보정 기준일:", TARGET_DATE.date())  # 보정 날짜
print("보정 출고수량:", NEED_QTY)  # 보정 수량
print("선택품목(비교용) 출고단가:", float(target_unit_price))  # 선택 단가
print("대체품목 후보 개수:", len(substitute_list))  # 후보 개수
print(substitute_list.head(30))  # 상위 30개 출력


ValueError: 선택품목 출고단가를 TARGET_DATE 기준으로 못 찾아서 단가 비교 불가

In [33]:
TARGET_DATE = pd.Timestamp("2025-01-02")  # 보정 기준일(1/2)
PRICE_TOL = 0.18  # 단가 허용폭(18%)

stock_col_submit = "본사실재고"  # 제출본 실재고
use_col_submit = "사용여부"  # 제출본 사용구분(과세/면세)
name_col_submit = "품목명"  # 제출본 품목명

df_submit_100["품목코드"] = df_submit_100["품목코드"].astype(str)  # 제출본 코드 문자열
df_suful_100["품목코드"] = df_suful_100["품목코드"].astype(str)  # 수불부 코드 문자열
df_submit_100[stock_col_submit] = pd.to_numeric(df_submit_100[stock_col_submit], errors="coerce").fillna(0).astype(float)  # 실재고 숫자화

target_row_submit = df_submit_100[df_submit_100["품목코드"] == str(target_item)].iloc[0]  # 선택품목 제출본 1행
target_stock_submit = float(target_row_submit[stock_col_submit])  # 선택품목 제출 실재고
target_use = str(target_row_submit[use_col_submit]).strip()  # 선택품목 사용구분
target_name = str(target_row_submit[name_col_submit]).strip()  # 선택품목 품목명

df_suful_100 = df_suful_100.sort_values(["품목코드", "일자"]).copy()  # 조회용 정렬

def get_stock_asof(item_code: str, date: pd.Timestamp) -> float:  # 기준일 재고(직전 포함)
    d = df_suful_100[df_suful_100["품목코드"] == item_code].copy()  # 품목 필터
    d = d[d["일자"] <= date].copy()  # 날짜 필터
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d["재고수량"], errors="coerce").fillna(np.nan).iloc[-1])  # 마지막 재고

def get_out_price_on_date(item_code: str, date: pd.Timestamp) -> float:  # 당일 유효 출고단가 평균
    d = df_suful_100[df_suful_100["품목코드"] == item_code].copy()  # 품목 필터
    d = d[d["일자"] == date].copy()  # 당일만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d["출고수량"] = pd.to_numeric(d["출고수량"], errors="coerce").fillna(0)  # 출고수량 숫자화
    d["출고단가"] = pd.to_numeric(d["출고단가"], errors="coerce")  # 출고단가 숫자화
    d = d[(d["출고수량"] > 0) & (d["출고단가"] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    return float(d["출고단가"].mean())  # 평균 단가

def get_nearest_valid_out_price_after(item_code: str, date: pd.Timestamp) -> float:  # 기준일 이후 가장 가까운 유효 출고단가
    d = df_suful_100[df_suful_100["품목코드"] == item_code].copy()  # 품목 필터
    d = d[d["일자"] >= date].copy()  # 기준일 이후만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d["출고수량"] = pd.to_numeric(d["출고수량"], errors="coerce").fillna(0)  # 출고수량 숫자화
    d["출고단가"] = pd.to_numeric(d["출고단가"], errors="coerce")  # 출고단가 숫자화
    d = d[(d["출고수량"] > 0) & (d["출고단가"] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    d = d.sort_values("일자").copy()  # 날짜 오름차순
    return float(d["출고단가"].iloc[0])  # 가장 가까운 단가

need_qty = None  # 보정 수량 변수

df_target = df_suful_100[df_suful_100["품목코드"] == str(target_item)].copy()  # 선택품목만
df_target = df_target.sort_values("일자").reset_index(drop=True)  # 날짜 정렬
df_target["is_minus"] = df_target["재고수량"] < 0  # 마이너스 여부
df_target["group"] = df_target["is_minus"].ne(df_target["is_minus"].shift()).cumsum()  # 구간 그룹
df_minus = df_target[df_target["is_minus"]].copy()  # 마이너스만

if len(df_minus) == 0:  # 마이너스가 없으면
    raise ValueError("선택품목에 중간 마이너스 구간이 없어")  # 중단

result = (  # 구간 요약
    df_minus.groupby("group")  # 구간별
    .agg(시작일=("일자", "min"), 종료일=("일자", "max"), 최대마이너스수=("재고수량", "min"))  # 집계
    .reset_index(drop=True)  # index 제거
)  # 요약 테이블

first_fix = result.sort_values("시작일").iloc[0]  # 첫 구간
TARGET_DATE = pd.Timestamp(first_fix["시작일"])  # 보정 기준일 재확정
NEED_QTY = int(abs(first_fix["최대마이너스수"]))  # 보정 수량 확정

target_unit_price = get_nearest_valid_out_price_after(str(target_item), TARGET_DATE)  # 선택품목 기준단가(이후 가장 가까운)
if pd.isna(target_unit_price):  # 못 찾으면
    raise ValueError("선택품목 기준단가(기준일 이후 유효 출고단가)를 못 찾아서 비교 불가")  # 중단

cand = df_submit_100.copy()  # 후보 시작
cand = cand[cand["품목코드"] != str(target_item)].copy()  # 자기 자신 제외
cand = cand[cand[stock_col_submit] > target_stock_submit].copy()  # (1) 제출 실재고 더 많음
cand = cand[cand[use_col_submit].astype(str).str.strip() == target_use].copy()  # (2) 사용구분 동일

cand["T_재고"] = cand["품목코드"].apply(lambda x: get_stock_asof(str(x), TARGET_DATE))  # (3) 기준일 재고
cand["T_추가출고후재고"] = cand["T_재고"] - NEED_QTY  # 기준일에 NEED_QTY 출고 가정
cand = cand[cand["T_추가출고후재고"] > 0].copy()  # (3) 출고 후 재고 0 초과

cand["T_출고단가"] = cand["품목코드"].apply(lambda x: get_out_price_on_date(str(x), TARGET_DATE))  # (4) 기준일 당일 단가
cand = cand[~pd.isna(cand["T_출고단가"])].copy()  # 단가 없는 후보 제외

cand["단가차이율"] = (cand["T_출고단가"] - float(target_unit_price)).abs() / float(target_unit_price)  # 차이율
cand = cand[cand["단가차이율"] <= PRICE_TOL].copy()  # (4) 18% 이내

substitute_list = cand[  # 결과 컬럼
    ["품목코드", name_col_submit, stock_col_submit, use_col_submit, "T_재고", "T_추가출고후재고", "T_출고단가", "단가차이율"]
].copy()  # 복사

substitute_list = substitute_list.sort_values(["단가차이율", stock_col_submit], ascending=[True, False]).reset_index(drop=True)  # 정렬

print("선택품목코드:", target_item)  # 선택품목코드
print("선택품목명:", target_name)  # 선택품목명
print("보정 기준일:", TARGET_DATE.date())  # 보정 날짜
print("보정 출고수량:", NEED_QTY)  # 보정 수량
print("선택품목 기준단가(가장 가까운 유효 출고단가):", float(target_unit_price))  # 기준단가
print("대체품목 후보 개수:", len(substitute_list))  # 후보 개수
print(substitute_list)  # 후보 리스트

선택품목코드: 3089
선택품목명: FS 냉동반다론(STEAM LAYER RICE CAKE)
보정 기준일: 2025-01-02
보정 출고수량: 11
선택품목 기준단가(가장 가까운 유효 출고단가): 90000.0
대체품목 후보 개수: 1
   품목코드                           품목명  본사실재고 사용여부  T_재고  T_추가출고후재고    T_출고단가  \
0  1651  피아 에이 스페셜 케익(A SPECIAL CAKE)  246.0   과세  19.0        8.0  105000.0   

      단가차이율  
0  0.166667  


In [34]:
# 대체품목이 없으면 중단
if len(substitute_list) == 0:  # 후보 0개면
    raise ValueError("대체품목 후보가 없어 이력/재계산을 할 수 없어")  # 중단

# 대체품목 1순위 선택
sub_item = str(substitute_list.iloc[0]["품목코드"])  # 대체품목코드
sub_name = str(substitute_list.iloc[0]["품목명"])  # 대체품목명

# 품목코드 문자열 통일
df_suful_100["품목코드"] = df_suful_100["품목코드"].astype(str)  # 코드 통일

# 날짜 정렬(안전)
df_suful_100 = df_suful_100.sort_values(["품목코드", "일자"]).copy()  # 정렬

def stock_asof(df: pd.DataFrame, item_code: str, date: pd.Timestamp) -> float:  # 기준일 재고(직전 포함)
    d = df[df["품목코드"] == str(item_code)].copy()  # 해당 품목만
    d = d[d["일자"] <= date].copy()  # 기준일 이전/당일
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d["재고수량"], errors="coerce").fillna(np.nan).iloc[-1])  # 마지막 재고

def pick_party(df: pd.DataFrame, item_code: str, date: pd.Timestamp) -> str:  # 기준일 거래처(가능하면 선택품목 당일)
    d = df[(df["품목코드"] == str(item_code)) & (df["일자"] == date)].copy()  # 당일만
    if len(d) == 0:  # 당일 행 없으면
        return "[대체이동]"  # 기본값
    v = str(d.iloc[0].get("거래처명", "")).strip()  # 거래처명
    return v if v != "" and v.lower() != "nan" else "[대체이동]"  # 빈값 처리

# 보정 전 재고(1/2 기준)
target_stock_before = stock_asof(df_suful_100, str(target_item), TARGET_DATE)  # 문제품목 재고(보정 전)
sub_stock_before = stock_asof(df_suful_100, sub_item, TARGET_DATE)  # 대체품목 재고(보정 전)

# 거래처(이력용)
party = pick_party(df_suful_100, str(target_item), TARGET_DATE)  # 거래처명

# 보정 후 대체품목 재고(1/2 기준)
sub_stock_after = sub_stock_before - float(NEED_QTY)  # 대체품목 이동 후 재고

# 이력 1줄 생성
df_move_log = pd.DataFrame([  # 이력 데이터프레임
    {
        "일자": TARGET_DATE,  # 일자
        "거래처": party,  # 거래처
        "문제품목코드": str(target_item),  # 문제품목코드
        "문제품목명": str(target_name),  # 문제품목명
        "문제품목재고량": target_stock_before,  # 문제품목재고량(보정 전)
        "대체품목코드": sub_item,  # 대체품목코드
        "대체품목명": sub_name,  # 대체품목명
        "대체품목재고량": sub_stock_before,  # 대체품목재고량(보정 전)
        "이동량": float(NEED_QTY),  # 이동량
        "대체품목이동후 재고량": sub_stock_after,  # 대체품목 이동 후 재고
    }
])  # 이력 1줄

# 보정용 데이터 복사(원본 보존)
df_suful_adj = df_suful_100.copy()  # 조정본

# 문제품목은 TARGET_DATE 이후 재고를 +NEED_QTY
mask_target = (df_suful_adj["품목코드"] == str(target_item)) & (df_suful_adj["일자"] >= TARGET_DATE)  # 문제품목 이후
df_suful_adj.loc[mask_target, "재고수량"] = pd.to_numeric(df_suful_adj.loc[mask_target, "재고수량"], errors="coerce") + float(NEED_QTY)  # 재고 +보정

# 대체품목은 TARGET_DATE 이후 재고를 -NEED_QTY
mask_sub = (df_suful_adj["품목코드"] == sub_item) & (df_suful_adj["일자"] >= TARGET_DATE)  # 대체품목 이후
df_suful_adj.loc[mask_sub, "재고수량"] = pd.to_numeric(df_suful_adj.loc[mask_sub, "재고수량"], errors="coerce") - float(NEED_QTY)  # 재고 -보정

# 조정본 정렬(보기용)
df_suful_adj = df_suful_adj.sort_values(["품목코드", "일자"]).reset_index(drop=True)  # 정렬

# 결과 출력
print("=== 대체 이력 1줄 ===")  # 구분 출력
print(df_move_log)  # 이력 출력
print("=== 조정된 수불부(df_suful_adj) 생성 완료 ===")  # 완료 출력
print("조정본 행 수:", len(df_suful_adj))  # 행 수 출력

=== 대체 이력 1줄 ===
          일자                   거래처 문제품목코드                            문제품목명  \
0 2025-01-02  [이동] 01본사창고 → 04지사창고   3089  FS 냉동반다론(STEAM LAYER RICE CAKE)   

   문제품목재고량 대체품목코드                         대체품목명  대체품목재고량   이동량  대체품목이동후 재고량  
0    -10.0   1651  피아 에이 스페셜 케익(A SPECIAL CAKE)     19.0  11.0          8.0  
=== 조정된 수불부(df_suful_adj) 생성 완료 ===
조정본 행 수: 34146


In [35]:
# 선택품목 조정 후 이력만 추출
df_chk = df_suful_adj[df_suful_adj["품목코드"] == str(target_item)].copy()  # 선택품목만
df_chk = df_chk.sort_values("일자").reset_index(drop=True)  # 정렬

# 마이너스 여부
df_chk["is_minus"] = df_chk["재고수량"] < 0  # 마이너스 여부

# 구간 그룹
df_chk["group"] = df_chk["is_minus"].ne(df_chk["is_minus"].shift()).cumsum()  # 구간 그룹

# 마이너스만
df_chk_minus = df_chk[df_chk["is_minus"]].copy()  # 마이너스만

# 마이너스 구간 요약
if len(df_chk_minus) == 0:  # 마이너스가 없으면
    print("조정 후: 중간 마이너스 구간 0개")  # 결과 출력
else:  # 마이너스가 있으면
    df_chk_result = (  # 요약
        df_chk_minus.groupby("group")  # 구간별
        .agg(시작일=("일자", "min"), 종료일=("일자", "max"), 최대마이너스수=("재고수량", "min"))  # 집계
        .reset_index(drop=True)  # 정리
    )  # 결과
    print("조정 후: 중간 마이너스 구간 개수:", len(df_chk_result))  # 구간 수
    print(df_chk_result)  # 요약 출력

조정 후: 중간 마이너스 구간 0개


In [36]:
OUT_MOVE_LOG = r"./대체이력이력.xlsx"  # 이력 저장 파일
OUT_SUFUL_ADJ = r"./재고수불부_보정본.xlsx"  # 보정본 저장 파일

# 후보가 없으면 중단
if len(substitute_list) == 0:  # 후보 0개면
    raise ValueError("대체품목 후보가 없어 저장할 수 없어")  # 중단

# 대체품목 1순위 선택
sub_item = str(substitute_list.iloc[0]["품목코드"])  # 대체품목코드
sub_name = str(substitute_list.iloc[0]["품목명"])  # 대체품목명

# 품목코드 문자열 통일
df_suful_100["품목코드"] = df_suful_100["품목코드"].astype(str)  # 수불부 코드 통일
df_submit_100["품목코드"] = df_submit_100["품목코드"].astype(str)  # 제출본 코드 통일

# 정렬(안전)
df_suful_100 = df_suful_100.sort_values(["품목코드", "일자"]).copy()  # 정렬

def stock_asof(df: pd.DataFrame, item_code: str, date: pd.Timestamp) -> float:  # 기준일 재고(직전 포함)
    d = df[df["품목코드"] == str(item_code)].copy()  # 해당 품목만
    d = d[d["일자"] <= date].copy()  # 기준일 이전/당일
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d["재고수량"], errors="coerce").fillna(np.nan).iloc[-1])  # 마지막 재고

def pick_party(df: pd.DataFrame, item_code: str, date: pd.Timestamp) -> str:  # 기준일 거래처(없으면 기본)
    d = df[(df["품목코드"] == str(item_code)) & (df["일자"] == date)].copy()  # 당일만
    if len(d) == 0:  # 당일 행 없으면
        return "[대체보정]"  # 기본
    v = str(d.iloc[0].get("거래처명", "")).strip()  # 거래처명 가져오기
    return v if v != "" and v.lower() != "nan" else "[대체보정]"  # 빈값 처리

# 보정 전 재고(기준일)
target_stock_before = stock_asof(df_suful_100, str(target_item), TARGET_DATE)  # 문제품목 재고(전)
sub_stock_before = stock_asof(df_suful_100, sub_item, TARGET_DATE)  # 대체품목 재고(전)

# 거래처(이력용)
party = pick_party(df_suful_100, str(target_item), TARGET_DATE)  # 거래처명

# 보정 후 재고(기준일 기준)
target_stock_after = target_stock_before + float(NEED_QTY)  # 문제품목 재고(후)
sub_stock_after = sub_stock_before - float(NEED_QTY)  # 대체품목 재고(후)

# 이력 1줄 생성
df_move_log = pd.DataFrame([  # 이력 테이블
    {
        "일자": TARGET_DATE,  # 일자
        "거래처": party,  # 거래처
        "문제품목코드": str(target_item),  # 문제품목코드
        "문제품목명": str(target_name),  # 문제품목명
        "문제품목재고량": target_stock_before,  # 문제품목재고량(전)
        "대체품목코드": sub_item,  # 대체품목코드
        "대체품목명": sub_name,  # 대체품목명
        "대체품목재고량": sub_stock_before,  # 대체품목재고량(전)
        "이동량": float(NEED_QTY),  # 이동량
        "대체품목이동후 재고량": sub_stock_after,  # 대체품목 이동후 재고
    }
])  # 이력 완료

# 보정본 생성(원본 보존)
df_suful_adj = df_suful_100.copy()  # 보정본 복사

# 문제품목은 기준일 이후 재고 +NEED_QTY
mask_target = (df_suful_adj["품목코드"] == str(target_item)) & (df_suful_adj["일자"] >= TARGET_DATE)  # 문제품목 이후
df_suful_adj.loc[mask_target, "재고수량"] = pd.to_numeric(df_suful_adj.loc[mask_target, "재고수량"], errors="coerce") + float(NEED_QTY)  # 재고 보정(+)

# 대체품목은 기준일 이후 재고 -NEED_QTY
mask_sub = (df_suful_adj["품목코드"] == sub_item) & (df_suful_adj["일자"] >= TARGET_DATE)  # 대체품목 이후
df_suful_adj.loc[mask_sub, "재고수량"] = pd.to_numeric(df_suful_adj.loc[mask_sub, "재고수량"], errors="coerce") - float(NEED_QTY)  # 재고 보정(-)

# 보정본 정렬
df_suful_adj = df_suful_adj.sort_values(["품목코드", "일자"]).reset_index(drop=True)  # 정렬

# 이력 저장
df_move_log.to_excel(OUT_MOVE_LOG, index=False)  # 이력 엑셀 저장

# 보정본 저장
df_suful_adj.to_excel(OUT_SUFUL_ADJ, index=False)  # 보정본 엑셀 저장

# 저장 결과 출력
print("이력 저장:", OUT_MOVE_LOG)  # 이력 파일 경로
print("보정본 저장:", OUT_SUFUL_ADJ)  # 보정본 파일 경로
print("문제품목코드:", target_item, "재고(전→후):", target_stock_before, "→", target_stock_after)  # 문제품목 변화
print("대체품목코드:", sub_item, "재고(전→후):", sub_stock_before, "→", sub_stock_after)  # 대체품목 변화

이력 저장: ./대체이력이력.xlsx
보정본 저장: ./재고수불부_보정본.xlsx
문제품목코드: 3089 재고(전→후): -10.0 → 1.0
대체품목코드: 1651 재고(전→후): 19.0 → 8.0


In [37]:
import pandas as pd  # 데이터 처리
import numpy as np  # 수치 처리

PRICE_TOL = 0.18  # 단가 허용폭(18%)

df_submit_100["품목코드"] = df_submit_100["품목코드"].astype(str)  # 제출본 코드 통일
df_suful_100["품목코드"] = df_suful_100["품목코드"].astype(str)  # 수불부 코드 통일
df_submit_100["본사실재고"] = pd.to_numeric(df_submit_100["본사실재고"], errors="coerce").fillna(0).astype(float)  # 실재고 숫자화

df_suful_100 = df_suful_100.sort_values(["품목코드", "일자"]).copy()  # 정렬

def min_stock_after(item_code: str, date: pd.Timestamp) -> float:  # 기준일 이후 최소 재고
    d = df_suful_100[df_suful_100["품목코드"] == str(item_code)].copy()  # 품목 필터
    d = d[d["일자"] >= date].copy()  # 기준일 이후만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    s = pd.to_numeric(d["재고수량"], errors="coerce")  # 재고수량 숫자화
    return float(s.min())  # 최소값

def get_out_price_on_date(item_code: str, date: pd.Timestamp) -> float:  # 당일 유효 출고단가 평균
    d = df_suful_100[df_suful_100["품목코드"] == str(item_code)].copy()  # 품목 필터
    d = d[d["일자"] == date].copy()  # 당일만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d["출고수량"] = pd.to_numeric(d["출고수량"], errors="coerce").fillna(0)  # 출고수량 숫자화
    d["출고단가"] = pd.to_numeric(d["출고단가"], errors="coerce")  # 출고단가 숫자화
    d = d[(d["출고수량"] > 0) & (d["출고단가"] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    return float(d["출고단가"].mean())  # 평균 단가

def get_nearest_valid_out_price_after(item_code: str, date: pd.Timestamp) -> float:  # 기준일 이후 가장 가까운 유효 출고단가
    d = df_suful_100[df_suful_100["품목코드"] == str(item_code)].copy()  # 품목 필터
    d = d[d["일자"] >= date].copy()  # 기준일 이후만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d["출고수량"] = pd.to_numeric(d["출고수량"], errors="coerce").fillna(0)  # 출고수량 숫자화
    d["출고단가"] = pd.to_numeric(d["출고단가"], errors="coerce")  # 출고단가 숫자화
    d = d[(d["출고수량"] > 0) & (d["출고단가"] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    d = d.sort_values("일자").copy()  # 날짜 정렬
    return float(d["출고단가"].iloc[0])  # 가장 가까운 단가

df_target = df_suful_100[df_suful_100["품목코드"] == str(target_item)].copy()  # 선택품목만
df_target = df_target.sort_values("일자").reset_index(drop=True)  # 정렬
df_target["is_minus"] = df_target["재고수량"] < 0  # 마이너스 여부
df_target["group"] = df_target["is_minus"].ne(df_target["is_minus"].shift()).cumsum()  # 구간 그룹
df_minus = df_target[df_target["is_minus"]].copy()  # 마이너스만

if len(df_minus) == 0:  # 마이너스 없으면
    raise ValueError("선택품목에 중간 마이너스 구간이 없어")  # 중단

result = (  # 구간 요약
    df_minus.groupby("group")  # 구간별
    .agg(시작일=("일자", "min"), 최대마이너스수=("재고수량", "min"))  # 최소만
    .reset_index(drop=True)  # 정리
)  # 요약

first_fix = result.sort_values("시작일").iloc[0]  # 첫 구간
TARGET_DATE = pd.Timestamp(first_fix["시작일"])  # 보정 기준일
NEED_QTY = int(abs(first_fix["최대마이너스수"]))  # 보정 수량

target_row_submit = df_submit_100[df_submit_100["품목코드"] == str(target_item)].iloc[0]  # 제출본 1행
target_stock_submit = float(target_row_submit["본사실재고"])  # 실재고
target_use = str(target_row_submit["사용여부"]).strip()  # 사용구분
target_name = str(target_row_submit["품목명"]).strip()  # 품목명

target_unit_price = get_nearest_valid_out_price_after(str(target_item), TARGET_DATE)  # 선택품목 기준단가
if pd.isna(target_unit_price):  # 없으면
    raise ValueError("선택품목 기준단가를 못 찾아서 비교 불가")  # 중단

cand = df_submit_100.copy()  # 후보 시작
cand = cand[cand["품목코드"] != str(target_item)].copy()  # 자기 제외
cand = cand[cand["본사실재고"] > target_stock_submit].copy()  # (1) 실재고 더 많음
cand = cand[cand["사용여부"].astype(str).str.strip() == target_use].copy()  # (2) 사용구분 동일

cand["T_출고단가"] = cand["품목코드"].apply(lambda x: get_out_price_on_date(str(x), TARGET_DATE))  # T일 단가
cand = cand[~pd.isna(cand["T_출고단가"])].copy()  # 단가 없는 후보 제외

cand["단가차이율"] = (cand["T_출고단가"] - float(target_unit_price)).abs() / float(target_unit_price)  # 차이율
cand = cand[cand["단가차이율"] <= PRICE_TOL].copy()  # (4) 18% 이내

cand["T이후_최소재고"] = cand["품목코드"].apply(lambda x: min_stock_after(str(x), TARGET_DATE))  # T이후 최소재고
cand["T이후_최소재고_이동후"] = cand["T이후_최소재고"] - float(NEED_QTY)  # 이동 후 최소재고 가정
cand = cand[cand["T이후_최소재고_이동후"] >= 0].copy()  # (추가) 이동 후 마이너스 금지

substitute_list = cand[  # 결과 컬럼
    ["품목코드", "품목명", "본사실재고", "사용여부", "T_출고단가", "단가차이율", "T이후_최소재고", "T이후_최소재고_이동후"]
].copy()  # 복사

substitute_list = substitute_list.sort_values(["단가차이율", "본사실재고"], ascending=[True, False]).reset_index(drop=True)  # 정렬

print("보정 기준일:", TARGET_DATE.date())  # 기준일
print("보정 수량:", NEED_QTY)  # 수량
print("단가 허용폭:", PRICE_TOL)  # 허용폭
print("대체품목 후보 개수:", len(substitute_list))  # 후보 수
print(substitute_list)  # 후보 출력


보정 기준일: 2025-01-02
보정 수량: 11
단가 허용폭: 0.18
대체품목 후보 개수: 0
Empty DataFrame
Columns: [품목코드, 품목명, 본사실재고, 사용여부, T_출고단가, 단가차이율, T이후_최소재고, T이후_최소재고_이동후]
Index: []


In [38]:
OUT_XLSX = r"./대체이력_보정수불부.xlsx"  # 결과 엑셀 파일
PRICE_TOL = 0.18  # 단가 허용폭(18%)

stock_col_submit = "본사실재고"  # 제출본 실재고
use_col_submit = "사용여부"  # 제출본 사용구분
name_col_submit = "품목명"  # 제출본 품목명

df_submit_100["품목코드"] = df_submit_100["품목코드"].astype(str)  # 제출본 코드 통일
df_suful_100["품목코드"] = df_suful_100["품목코드"].astype(str)  # 수불부 코드 통일
df_submit_100[stock_col_submit] = pd.to_numeric(df_submit_100[stock_col_submit], errors="coerce").fillna(0).astype(float)  # 실재고 숫자화

df_suful_100 = df_suful_100.sort_values(["품목코드", "일자"]).copy()  # 정렬

def get_out_price_on_date(item_code: str, date: pd.Timestamp) -> float:  # 당일 유효 출고단가 평균
    d = df_suful_100[df_suful_100["품목코드"] == str(item_code)].copy()  # 품목 필터
    d = d[d["일자"] == date].copy()  # 당일만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d["출고수량"] = pd.to_numeric(d["출고수량"], errors="coerce").fillna(0)  # 출고수량 숫자화
    d["출고단가"] = pd.to_numeric(d["출고단가"], errors="coerce")  # 출고단가 숫자화
    d = d[(d["출고수량"] > 0) & (d["출고단가"] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    return float(d["출고단가"].mean())  # 평균 단가

def min_stock_after(item_code: str, date: pd.Timestamp) -> float:  # 기준일 이후 최소 재고
    d = df_suful_100[df_suful_100["품목코드"] == str(item_code)].copy()  # 품목 필터
    d = d[d["일자"] >= date].copy()  # 기준일 이후만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    s = pd.to_numeric(d["재고수량"], errors="coerce")  # 재고수량 숫자화
    return float(s.min())  # 최소 재고

def stock_asof(item_code: str, date: pd.Timestamp) -> float:  # 기준일 재고(직전 포함)
    d = df_suful_100[df_suful_100["품목코드"] == str(item_code)].copy()  # 품목 필터
    d = d[d["일자"] <= date].copy()  # 기준일 이전/당일
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d["재고수량"], errors="coerce").fillna(np.nan).iloc[-1])  # 마지막 재고

def pick_party(item_code: str, date: pd.Timestamp) -> str:  # 이력용 거래처
    d = df_suful_100[(df_suful_100["품목코드"] == str(item_code)) & (df_suful_100["일자"] == date)].copy()  # 당일만
    if len(d) == 0:  # 없으면
        return "[대체보정]"  # 기본
    v = str(d.iloc[0].get("거래처명", "")).strip()  # 거래처명
    return v if v != "" and v.lower() != "nan" else "[대체보정]"  # 빈값 처리

target_row = df_submit_100[df_submit_100["품목코드"] == str(target_item)].iloc[0]  # 선택품목 제출본 1행
target_stock_submit = float(target_row[stock_col_submit])  # 선택품목 제출 실재고
target_use = str(target_row[use_col_submit]).strip()  # 선택품목 사용구분

cand = df_submit_100.copy()  # 후보 시작
cand = cand[cand["품목코드"] != str(target_item)].copy()  # 자기 제외
cand = cand[cand[stock_col_submit] > target_stock_submit].copy()  # (1) 제출 실재고 더 많음
cand = cand[cand[use_col_submit].astype(str).str.strip() == target_use].copy()  # (2) 사용구분 동일

cand["T_출고단가"] = cand["품목코드"].apply(lambda x: get_out_price_on_date(str(x), TARGET_DATE))  # (4) 1/2 출고단가
cand = cand[~pd.isna(cand["T_출고단가"])].copy()  # 단가 없는 후보 제외

cand["단가차이율"] = (cand["T_출고단가"] - float(target_unit_price)).abs() / float(target_unit_price)  # 단가차이율
cand = cand[cand["단가차이율"] <= PRICE_TOL].copy()  # 단가 18% 이내만

cand["T이후_최소재고"] = cand["품목코드"].apply(lambda x: min_stock_after(str(x), TARGET_DATE))  # 1/2 이후 최소재고
cand = cand[~pd.isna(cand["T이후_최소재고"])].copy()  # 최소재고 없는 후보 제외

cand["가용이동량"] = cand["T이후_최소재고"].apply(lambda x: int(np.floor(max(0, x))))  # 마이너스 없이 뺄 수 있는 최대량
cand = cand[cand["가용이동량"] > 0].copy()  # 1개라도 이동 가능한 후보만

cand = cand.sort_values(["단가차이율", "가용이동량"], ascending=[True, False]).reset_index(drop=True)  # 좋은 후보 우선 정렬

need = int(NEED_QTY)  # 남은 이동량
moves = []  # 분할 이동 결과 리스트

for i in range(len(cand)):  # 후보 순회
    if need <= 0:  # 다 채웠으면
        break  # 종료
    code = str(cand.loc[i, "품목코드"])  # 후보 코드
    name = str(cand.loc[i, name_col_submit])  # 후보 품목명
    avail = int(cand.loc[i, "가용이동량"])  # 가용 이동량
    move_qty = int(min(need, avail))  # 이번 품목에 배정할 이동량
    if move_qty <= 0:  # 0이면
        continue  # 스킵
    moves.append({"대체품목코드": code, "대체품목명": name, "이동량": move_qty})  # 이동 기록
    need -= move_qty  # 남은 이동량 감소

if need > 0:  # 아직 남았으면
    raise ValueError(f"분할 이동으로도 수량을 못 채워(남은수량={need}) -> 후보를 더 늘리거나(샘플100 말고 전체) 단가조건/기준을 조정해야해")  # 중단

party = pick_party(str(target_item), TARGET_DATE)  # 거래처
target_stock_before = stock_asof(str(target_item), TARGET_DATE)  # 문제품목 재고(전)

df_move_log_rows = []  # 이력 여러 줄 담기

for m in moves:  # 이동 내역 순회
    sub_code = str(m["대체품목코드"])  # 대체품목코드
    sub_name = str(m["대체품목명"])  # 대체품목명
    qty = float(m["이동량"])  # 이동량
    sub_stock_before = stock_asof(sub_code, TARGET_DATE)  # 대체품목 재고(전)
    sub_stock_after = sub_stock_before - qty  # 대체품목 재고(후)
    df_move_log_rows.append({  # 이력 1줄
        "일자": TARGET_DATE,  # 일자
        "거래처": party,  # 거래처
        "문제품목코드": str(target_item),  # 문제품목코드
        "문제품목명": str(target_name),  # 문제품목명
        "문제품목재고량": target_stock_before,  # 문제품목재고량(전)
        "대체품목코드": sub_code,  # 대체품목코드
        "대체품목명": sub_name,  # 대체품목명
        "대체품목재고량": sub_stock_before,  # 대체품목재고량(전)
        "이동량": qty,  # 이동량
        "대체품목이동후 재고량": sub_stock_after,  # 대체품목이동후 재고량
    })  # 이력 추가

df_move_log = pd.DataFrame(df_move_log_rows)  # 이력 DF 생성

df_suful_adj = df_suful_100.copy()  # 보정본 생성

mask_target = (df_suful_adj["품목코드"] == str(target_item)) & (df_suful_adj["일자"] >= TARGET_DATE)  # 문제품목 이후
df_suful_adj.loc[mask_target, "재고수량"] = pd.to_numeric(df_suful_adj.loc[mask_target, "재고수량"], errors="coerce") + float(NEED_QTY)  # 문제품목 재고 +11

for m in moves:  # 각 대체품목 보정
    sub_code = str(m["대체품목코드"])  # 대체품목코드
    qty = float(m["이동량"])  # 이동량
    mask_sub = (df_suful_adj["품목코드"] == sub_code) & (df_suful_adj["일자"] >= TARGET_DATE)  # 대체품목 이후
    df_suful_adj.loc[mask_sub, "재고수량"] = pd.to_numeric(df_suful_adj.loc[mask_sub, "재고수량"], errors="coerce") - qty  # 대체품목 재고 -qty

df_suful_adj = df_suful_adj.sort_values(["품목코드", "일자"]).reset_index(drop=True)  # 정렬

ok_all = True  # 검증 플래그
bad_list = []  # 문제 리스트

for m in moves:  # 검증 순회
    sub_code = str(m["대체품목코드"])  # 대체품목코드
    d = df_suful_adj[(df_suful_adj["품목코드"] == sub_code) & (df_suful_adj["일자"] >= TARGET_DATE)].copy()  # 이후만
    smin = float(pd.to_numeric(d["재고수량"], errors="coerce").min()) if len(d) > 0 else np.nan  # 최소 재고
    if pd.isna(smin) or smin < 0:  # 마이너스면
        ok_all = False  # 실패
        bad_list.append((sub_code, smin))  # 기록

if ok_all == False:  # 실패면
    raise ValueError(f"보정 후에도 대체품목 마이너스가 발생했어: {bad_list}")  # 중단

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:  # 엑셀 저장
    df_move_log.to_excel(writer, sheet_name="대체이력", index=False)  # 이력 시트
    df_suful_adj.to_excel(writer, sheet_name="재고수불부_보정본", index=False)  # 보정본 시트

print("보정 기준일:", TARGET_DATE.date())  # 기준일 출력
print("보정 수량:", NEED_QTY)  # 보정수량 출력
print("분할 이동 개수:", len(moves))  # 분할 건수 출력
print("분할 이동 내역:", moves)  # 분할 내역 출력
print("저장 완료:", OUT_XLSX)  # 저장 경로 출력

ValueError: 분할 이동으로도 수량을 못 채워(남은수량=11) -> 후보를 더 늘리거나(샘플100 말고 전체) 단가조건/기준을 조정해야해